In [6]:
# Imports
from PIL import Image
import torch
from torchvision import transforms
from IPython.display import display

import sys
sys.path.insert(0, "../")
from models.birefnet import BiRefNet


# Load Model
# Option 2 and Option 3 is better for local running -- we can modify codes locally.

# # # Option 1: loading BiRefNet with weights:
# from transformers import AutoModelForImageSegmentation
# birefnet = AutoModelForImageSegmentation.from_pretrained('zhengpeng7/BiRefNet', trust_remote_code=True)

# Option-2: loading weights with BiReNet codes:
model = [
        'zhengpeng7/BiRefNet',
        'zhengpeng7/BiRefNet-portrait',
        'zhengpeng7/BiRefNet-legacy', 'zhengpeng7/BiRefNet-DIS5K-TR_TEs', 'zhengpeng7/BiRefNet-DIS5K', 'zhengpeng7/BiRefNet-HRSOD', 'zhengpeng7/BiRefNet-COD',
        'zhengpeng7/BiRefNet_lite',     # Modify the `bb` in `config.py` to `swin_v1_tiny`.
    ][0]
birefnet = BiRefNet.from_pretrained(
    model
)
model_name = model.split('/')[-1]

# # Option-3: Loading model and weights from local disk:
# from utils import check_state_dict

# birefnet = BiRefNet(bb_pretrained=False)
# state_dict = torch.load('../BiRefNet-general-epoch_244.pth', map_location='cpu', weights_only=True)
# state_dict = check_state_dict(state_dict)
# birefnet.load_state_dict(state_dict)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.set_float32_matmul_precision(['high', 'highest'][0])

birefnet.to(device)
birefnet.eval()
print('BiRefNet is ready to use.')

# Input Data
transform_image = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

c:\Users\kasir\Miniconda3\envs\torch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BiRefNet is ready to use.


In [13]:

video_src_paths = 'power_hero.mp4'

In [15]:
import cv2
import numpy as np
from image_proc import refine_foreground
from time import time

autocast_ctx = torch.amp.autocast(device_type='cuda', dtype=[torch.float16, torch.bfloat16][0])
for video_src_path in video_src_paths[:]:
    print('\nvideo_src_path:', video_src_path)
    src_dir = os.path.join('frames-{}-video_{}'.format(model_name, os.path.splitext(os.path.basename(video_src_path))[0]))
    video_ext = os.path.splitext(video_src_path)[-1]
    video_dst_path_mask = video_src_path.replace(video_ext, '-preds_mask-{}'.format(model_name)+video_ext)
    video_dst_path_subject = video_src_path.replace(video_ext, '-preds_subject-{}'.format(model_name)+video_ext)
    vidcap = cv2.VideoCapture(video_src_path)
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    success, image = vidcap.read()

    video_writer_shape = image.shape[:2][::-1]
    video_writer_mask = cv2.VideoWriter(video_dst_path_mask, cv2.VideoWriter_fourcc(*'mp4v'), fps, video_writer_shape, isColor=False)
    video_writer_subject = cv2.VideoWriter(video_dst_path_subject, cv2.VideoWriter_fourcc(*'mp4v'), fps, video_writer_shape, isColor=True)

    count = 0
    while success:
        os.makedirs(src_dir, exist_ok=True)
        cv2.imwrite(os.path.join(src_dir, 'frame_{}.png'.format(count)), image)
        success, image = vidcap.read()
        count += 1


video_src_path: p


AttributeError: 'NoneType' object has no attribute 'shape'

In [17]:
import os
import cv2
import numpy as np
import torch
from image_proc import refine_foreground
from time import time

# Define your video paths and model name
video_src_paths = ['power_hero.mp4']  # Replace with your video(s)
model_name = 'BiRefNet'

autocast_ctx = torch.amp.autocast(device_type='cuda', dtype=torch.float16)

for video_src_path in video_src_paths:
    print('\nvideo_src_path:', video_src_path)

    src_dir = os.path.join('frames-{}-video_{}'.format(
        model_name, os.path.splitext(os.path.basename(video_src_path))[0]
    ))
    video_ext = os.path.splitext(video_src_path)[-1]
    video_dst_path_mask = video_src_path.replace(
        video_ext, f'-preds_mask-{model_name}{video_ext}')
    video_dst_path_subject = video_src_path.replace(
        video_ext, f'-preds_subject-{model_name}{video_ext}')

    vidcap = cv2.VideoCapture(video_src_path)
    fps = vidcap.get(cv2.CAP_PROP_FPS)
    success, image = vidcap.read()

    if not success or image is None:
        print(f"❌ Could not read video: {video_src_path}")
        continue

    video_writer_shape = image.shape[:2][::-1]
    video_writer_mask = cv2.VideoWriter(
        video_dst_path_mask, cv2.VideoWriter_fourcc(*'mp4v'),
        fps, video_writer_shape, isColor=False
    )
    video_writer_subject = cv2.VideoWriter(
        video_dst_path_subject, cv2.VideoWriter_fourcc(*'mp4v'),
        fps, video_writer_shape, isColor=True
    )

    os.makedirs(src_dir, exist_ok=True)

    count = 0
    while success and image is not None:
        frame_path = os.path.join(src_dir, f'frame_{count}.png')
        cv2.imwrite(frame_path, image)
        success, image = vidcap.read()
        count += 1



video_src_path: power_hero.mp4
